### Прогноз цен на недвижимость

Курсовой проект по машинному обучению (4 курс, 1 семестр).
Авторы: Эрнест и Иршат.
Датасет: объявления о продаже недвижимости с портала Zingat.
Цель: исследовать данные, подготовить признаки и обучить модели регрессии для прогнозирования стоимости объектов.


In [ ]:
# подключаем библиотеки для работы с данными, графиками и ML
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## Этап 1. Исследование данных (EDA)

### 1.1 Знакомство с данными
Загружаем датасет, смотрим его размерность, первые, последние и случайные строки.


In [ ]:
# читаем датасет из папки data
df = pd.read_csv("../data/real_estate_data.csv", low_memory=False)
print("Размер датасета (строк, колонок):", df.shape)
# выводим первые 5 строк
df.head()

In [ ]:
# смотрим последние 5 строк
df.tail()

In [ ]:
# выводим одну случайную строку для проверки разнообразия данных
df.sample(1, random_state=42)

### Описание признаков датасета (по документации Zingat)

* **id**: уникальный идентификатор объявления
* **type**: тип объекта (Konut - жилье)
* **sub_type**: подтип жилья (Daire - квартира, Villa - вилла, Rezidans - резиденция)
* **start_date** / **end_date**: даты публикации и снятия с сайта
* **listing_type**: тип сделки (1 - продажа, 2 - аренда)
* **tom**: дней на рынке (time on market)
* **building_age**: возраст строения (0, 1, 2, 6-10 arasi и т.д.)
* **total_floor_count**: этажность здания
* **floor_no**: этаж квартиры
* **room_count**: число комнат (формат 2+1, 3+1)
* **size**: общая площадь в м2
* **address**: город / район / микрорайон
* **furnished**: меблировка (100% пропусков)
* **heating_type**: система отопления (Kombi, Merkezi Sistem и др.)
* **price**: цена объекта в турецких лирах (целевая переменная)
* **price_currency**: валюта (TRY)


In [ ]:
# типы данных и ненулевые значения
df.info()

### 1.2 Описательная статистика и анализ целевой переменной


In [ ]:
# базовые статистики числовых колонок: минимум, максимум, среднее, квантили
df.describe()

In [ ]:
# статистика категориальных признаков: количество уникальных и самый частый класс
df.describe(include='O')

In [ ]:
# проверяем скошенность (асимметрию) цены:
# положительный коэффициент говорит о сильном хвосте вправо (дорогие элитные объекты)
print("Коэффициент асимметрии цены:", df['price'].skew())
print("Асимметрия после log1p:", np.log1p(df['price'].dropna()).skew())

### 1.3 Визуальный анализ данных (5+ типов графиков)


In [ ]:
# 1. Гистограммы распределения всех числовых признаков
df.hist(figsize=(20, 10))
plt.show()

In [ ]:
# 2. Круговая диаграмма долей подтипов недвижимости
plt.figure(figsize=(8, 6))
df['sub_type'].value_counts().head(5).plot(kind='pie', autopct='%.2f', textprops={'color': 'black'})
plt.title('Доли подтипов недвижимости', fontsize=14)
plt.ylabel('')
plt.show()

Больше 85% всех объявлений в датасете - это квартиры (Daire).


In [ ]:
# 3. Boxplot: распределение цены по подтипам недвижимости
top_subtypes = df['sub_type'].value_counts().head(5).index
plt.figure(figsize=(12, 5))
sns.boxplot(data=df[df['sub_type'].isin(top_subtypes)], x='sub_type', y='price')
plt.title('Sub Type Vs Price', fontsize=15)
plt.xlabel('Тип жилья')
plt.ylabel('Цена (TRY)')
plt.ylim(0, 3000000)
plt.show()

Виллы и резиденции стоят ощутимо дороже типовых квартир.


In [ ]:
# 4. Столбчатая диаграмма средних цен по городам
df['city'] = df['address'].apply(lambda x: str(x).split('/')[0] if '/' in str(x) else 'Diger')
city_mean = df.groupby('city')['price'].mean().sort_values(ascending=False).head(8)

plt.figure(figsize=(12, 5))
sns.barplot(x=city_mean.index, y=city_mean.values)
plt.title('City Vs Average Price', fontsize=15)
plt.xlabel('Город')
plt.ylabel('Средняя цена (TRY)')
plt.show()

Стамбул и курортные города (Мугла, Анталья) лидируют по средней стоимости жилья.


In [ ]:
# 5. Scatterplot (точечная диаграмма): зависимость цены от площади
sample_data = df[(df['size'] >= 30) & (df['size'] <= 300) & (df['price'] <= 5000000)].sample(3000, random_state=42)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=sample_data, x='size', y='price', alpha=0.3)
plt.title('Size Vs Price', fontsize=15)
plt.xlabel('Площадь (м2)')
plt.ylabel('Цена (TRY)')
plt.show()

Видна прямая зависимость: чем больше площадь квартиры, тем выше цена.


### 1.4 Анализ пропусков, дубликатов и аномалий


In [ ]:
# таблица пропусков по колонкам
null_info = pd.DataFrame({
    'Пропусков': df.isnull().sum(),
    'Процент': (df.isnull().sum() / len(df) * 100).round(2)
})
null_info[null_info['Пропусков'] > 0]

In [ ]:
# проверяем наличие полных дубликатов строк
print("Количество полных дубликатов:", df.duplicated().sum())

# проверяем нереалистичные значения (цена <= 0, площадь <= 0)
bad_prices = (df['price'] <= 0).sum()
bad_sizes = (df['size'] <= 0).sum()
print("Записей с нулевой или отрицательной ценой:", bad_prices)
print("Записей с нулевой или отрицательной площадью:", bad_sizes)

### 1.5 Промежуточные выводы и гипотезы EDA

На основе анализа формулируем 4 гипотезы для последующего моделирования:
1. **Гипотеза о площади**: площадь объекта (size) сильнее всего положительно коррелирует со стоимостью.
2. **Гипотеза о локации**: город и район являются вторым ключевым фактором цены (Стамбул дороже других регионов).
3. **Гипотеза о комнатах**: количество комнат (room_count) увеличивает цену объекта, так как растет полезная площадь.
4. **Гипотеза об отоплении**: наличие современного газового отопления (Kombi) повышает ликвидность и цену квартиры.


## Этап 2. Предварительная обработка данных

Включает:
- фильтрацию только объявлений о продаже в лирах TRY;
- удаление неинформативной колонки furnished (100% пропусков);
- очистку выбросов по цене и площади;
- Feature Engineering (извлечение числа комнат, возраста здания);
- разделение на Train и Test ДО кодирования и масштабирования (исключение утечки данных!).


In [ ]:
# фильтруем только продажу (listing_type == 1) и валюту TRY
df_clean = df[df['listing_type'] == 1].copy()
df_clean = df_clean[(df_clean['price_currency'] == 'TRY') | (df_clean['price_currency'].isna())].copy()

# удаляем пустой столбец furnished (в нем 100% пропусков)
df_clean = df_clean.drop(columns=['furnished'], errors='ignore')

# удаляем пропуски в цене и площади
df_clean = df_clean.dropna(subset=['price', 'size'])

# убираем явные выбросы: цены меньше 20к или больше 6 млн, площадь меньше 20 или больше 400 м2
df_clean = df_clean[(df_clean['price'] >= 20000) & (df_clean['price'] <= 6000000)]
df_clean = df_clean[(df_clean['size'] >= 20) & (df_clean['size'] <= 400)]
print("Размер данных после очистки выбросов:", df_clean.shape)

In [ ]:
# Feature Engineering:
# 1. Извлекаем город из адреса
df_clean['city'] = df_clean['address'].apply(lambda x: str(x).split('/')[0] if '/' in str(x) else 'Diger')

# 2. Переводим категориальный возраст здания в числовой признак
age_dict = {
    '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5,
    '6-10 arası': 8, '11-15 arası': 13, '16-20 arası': 18,
    '21-25 arası': 23, '26-30 arası': 28, '31 ve üzeri': 35
}
df_clean['building_age_num'] = df_clean['building_age'].map(age_dict).fillna(10)

# 3. Извлекаем число спален из формата 2+1
def parse_rooms(val):
    val_str = str(val).strip()
    if '+' in val_str:
        parts = val_str.split('+')
        try:
            return int(parts[0]) + int(parts[1])
        except:
            return 3
    return 3

df_clean['total_rooms'] = df_clean['room_count'].apply(parse_rooms)

# смотрим созданные признаки
df_clean[['price', 'size', 'total_rooms', 'building_age_num', 'city', 'sub_type', 'heating_type']].head()

In [ ]:
# смотрим матрицу корреляций числовых признаков
numeric_cols = ['size', 'total_rooms', 'building_age_num', 'tom', 'price']
corr = df_clean[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='RdYlGn', center=0)
plt.title('Матрица корреляций числовых признаков', fontsize=14)
plt.show()

### 2.6 Разбиение выборки на Train и Test (без утечки данных)

ВАЖНО: Разделяем данные на Train (80%) и Test (20%) ДО кодирования категорий и масштабирования.
Обучение препроцессоров (LabelEncoder, StandardScaler, MinMaxScaler) будет происходить СТРОГО на обучающей выборке.


In [ ]:
# отбираем признаки для моделей
features = ['size', 'total_rooms', 'building_age_num', 'sub_type', 'city', 'heating_type', 'tom']
X = df_clean[features].copy()
y = df_clean['price'].copy()

# фиксируем random_state для воспроизводимости
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Размер обучающей выборки:", X_train.shape)
print("Размер тестовой выборки:", X_test.shape)

In [ ]:
# кодируем категориальные признаки
# обучаем кодировщик только на X_train, а X_test только трансформируем (защита от утечки данных!)
from sklearn.preprocessing import OrdinalEncoder

cat_cols = ['sub_type', 'city', 'heating_type']
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols].astype(str))
X_test[cat_cols] = encoder.transform(X_test[cat_cols].astype(str))

X_train.head()

## Этап 3. Построение моделей регрессии

Обучаем 3 модели разной природы:
1. **Линейная регрессия (Linear Regression)**
2. **Одиночное дерево решений (DecisionTreeRegressor)**
3. **Ансамблевая модель (RandomForestRegressor)**


In [ ]:
# функции для расчета метрик: MAE, MSE, RMSE, R2
from sklearn import metrics

def print_metrics(true, predicted):
    mae = metrics.mean_absolute_error(true, predicted)
    mse = metrics.mean_squared_error(true, predicted)
    rmse = np.sqrt(metrics.mean_squared_error(true, predicted))
    r2 = metrics.r2_score(true, predicted)
    mape = np.mean(np.abs((true - predicted) / true)) * 100
    print(f"MAE:  {mae:,.0f} TRY")
    print(f"MSE:  {mse:,.0f}")
    print(f"RMSE: {rmse:,.0f} TRY")
    print(f"MAPE: {mape:.2f}%")
    print(f"R2:   {r2:.4f}")

def get_metrics_row(model_name, true, predicted):
    mae = metrics.mean_absolute_error(true, predicted)
    mse = metrics.mean_squared_error(true, predicted)
    rmse = np.sqrt(metrics.mean_squared_error(true, predicted))
    r2 = metrics.r2_score(true, predicted)
    mape = np.mean(np.abs((true - predicted) / true)) * 100
    return [model_name, mae, mse, rmse, mape, r2]

### 3.1 Модель 1: Линейная регрессия


In [ ]:
from sklearn.linear_model import LinearRegression

lrm = LinearRegression()
lrm.fit(X_train, y_train)

test_pred_lrm = lrm.predict(X_test)
train_pred_lrm = lrm.predict(X_train)

print('Test: Линейная регрессия\n_____________________________________')
print_metrics(y_test, test_pred_lrm)

results_df = pd.DataFrame(
    [get_metrics_row("Linear Regression", y_test, test_pred_lrm)],
    columns=['Model', 'MAE', 'MSE', 'RMSE', 'MAPE (%)', 'R2 Square']
)

### 3.2 Модель 2: Дерево решений (Decision Tree)


In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=42, max_depth=8)
tree.fit(X_train, y_train)

test_pred_tree = tree.predict(X_test)

print('Test: Дерево решений\n_____________________________________')
print_metrics(y_test, test_pred_tree)

results_df2 = pd.DataFrame(
    [get_metrics_row("Decision Tree", y_test, test_pred_tree)],
    columns=['Model', 'MAE', 'MSE', 'RMSE', 'MAPE (%)', 'R2 Square']
)
results_df = pd.concat([results_df, results_df2], ignore_index=True)

### 3.3 Модель 3: Случайный лес (Random Forest)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# для скорости берем 50 деревьев и max_depth 12
rf = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

test_pred_rf = rf.predict(X_test)

print('Test: Random Forest\n_____________________________________')
print_metrics(y_test, test_pred_rf)

results_df2 = pd.DataFrame(
    [get_metrics_row("Random Forest", y_test, test_pred_rf)],
    columns=['Model', 'MAE', 'MSE', 'RMSE', 'MAPE (%)', 'R2 Square']
)
results_df = pd.concat([results_df, results_df2], ignore_index=True)